In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from darts import TimeSeries
from darts.models import NBEATSModel
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error
from statsmodels.tsa.statespace.sarimax import SARIMAX


# Load dataset (download from Kaggle first)
df = pd.read_csv('continuous dataset.csv')
df['datetime'] = pd.to_datetime(df['datetime'])
df = df.set_index('datetime').sort_index()

# Check hourly frequency and fill gaps
df = df.asfreq('h')
df['nat_demand'] = df['nat_demand'].interpolate()

df['hour'] = df.index.hour
df['dayofweek'] = df.index.dayofweek
df['month'] = df.index.month
df['is_weekend'] = (df['dayofweek'] >= 5).astype(int)

train = df.loc[:'2019-12-31']
test = df.loc['2020-01-01':]

model = SARIMAX(train['nat_demand'], order=(2,1,2), seasonal_order=(1,1,1,24))
sarima_fit = model.fit()
forecast_sarima = sarima_fit.forecast(len(test))

mae = mean_absolute_error(test['nat_demand'], forecast_sarima)
mape = mean_absolute_percentage_error(test['nat_demand'], forecast_sarima) * 100
print(f"MAE: {mae:.2f}, MAPE: {mape:.2f}%")

plt.figure(figsize=(14,6))
plt.plot(train.index, train['nat_demand'], label='Train', color='black')
plt.plot(test.index, test['nat_demand'], label='Test', color='gray')
plt.plot(test.index, forecast_sarima, label='SARIMA Forecast', color='blue')
plt.xlabel('Date')
plt.ylabel('Load (MW)')
plt.legend()
plt.title('Short-Term Electric Load Forecasting (SARIMA)')
plt.savefig('electric_load_forecast.png')
plt.show()

series = TimeSeries.from_dataframe(df, value_cols='nat_demand')
train, val = series.split_after('2021-12-31')

model = NBEATSModel(input_chunk_length=48, output_chunk_length=24, n_epochs=50)
model.fit(train)
forecast_nbeats = model.predict(len(val))


/usr/local/lib/python3.11/dist-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


In [2]:
! pip install darts

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.0/56.0 kB 2.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.3/46.3 kB 3.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 20.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.6/200.6 kB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 825.4/825.4 kB 45.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.7/37.7 MB 47.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 340.0/340.0 kB 24.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.2/87.2 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 48.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 38.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8

In [3]:
import pandas as pd
import matplotlib.pyplot as plt
from darts import TimeSeries
from darts.models import NBEATSModel
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error


FileNotFoundError: [Errno 2] No such file or directory: 'continuous dataset.csv'

In [5]:

# Load dataset
df = pd.read_csv('continuous dataset.csv')
df['datetime'] = pd.to_datetime(df['datetime'])
df = df.set_index('datetime').sort_index()

# Ensure hourly frequency and fill gaps
df = df.asfreq('h')
df['nat_demand'] = df['nat_demand'].interpolate()

# Convert to Darts TimeSeries
series = TimeSeries.from_dataframe(df, value_cols='nat_demand')

# Split data using a pandas.Timestamp
split_point = pd.Timestamp('2020-01-01')
train, test = series.split_after(split_point)

# Fit N-BEATS model
model = NBEATSModel(input_chunk_length=48, output_chunk_length=24, n_epochs=5, random_state=3363)
model.fit(train)

# Forecast
forecast_nbeats = model.predict(len(test))

# Compute error metrics
mae = mean_absolute_error(test.values(), forecast_nbeats.values())
mape = mean_absolute_percentage_error(test.values(), forecast_nbeats.values()) * 100
print(f"MAE: {mae:.2f}, MAPE: {mape:.2f}%")

# Plot results
plt.figure(figsize=(14,6))
train.plot(label='Train', lw=1, color='black')
test.plot(label='Test', lw=1, color='gray')
forecast_nbeats.plot(label='N-BEATS Forecast', lw=2, color='blue')
plt.xlabel('Date')
plt.ylabel('Load (MW)')
plt.title('Short-Term Electric Load Forecasting (N-BEATS)')
plt.legend()
plt.savefig('electric_load_forecast_nbeats.png')
plt.show()


INFO:pytorch_lightning.utilities.rank_zero:GPU available: False, used: False
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:HPU available: False, using: 0 HPUs
INFO:pytorch_lightning.callbacks.model_summary:
  | Name            | Type             | Params | Mode 
-------------------------------------------------------------
0 | criterion       | MSELoss          | 0      | train
1 | train_criterion | MSELoss          | 0      | train
2 | val_criterion   | MSELoss          | 0      | train
3 | train_metrics   | MetricCollection | 0      | train
4 | val_metrics     | MetricCollection | 0      | train
5 | stacks          | ModuleList       | 6.4 M  | train
-------------------------------------------------------------
6.4 M     Trainable params
1.6 K     Non-trainable params
6.4 M     Total params
25.551    Total estimated model params size (MB)
396       Modules in train mode
0         Modules in eval mode


Training: |          | 0/? [00:00<?, ?it/s]

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=5` reached.
INFO:pytorch_lightning.utilities.rank_zero:GPU available: False, used: False
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:HPU available: False, using: 0 HPUs


Predicting: |          | 0/? [00:00<?, ?it/s]

MAE: 281.45, MAPE: 24.93%


In [7]:
import pandas as pd
import matplotlib.pyplot as plt
from darts import TimeSeries
from darts.models import NBEATSModel
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error
from darts.dataprocessing.transformers import Scaler
from darts.utils.timeseries_generation import datetime_attribute_timeseries

# Load and preprocess data
df = pd.read_csv('continuous dataset.csv')
df['datetime'] = pd.to_datetime(df['datetime'])
df = df.set_index('datetime').asfreq('h').sort_index()

# Target series
series = TimeSeries.from_dataframe(df, value_cols='nat_demand')

# Covariates: weather (past) and calendar (future)
weather_cols = ['T2M_toc','QV2M_toc','TQL_toc','W2M_toc',
                'T2M_san','QV2M_san','TQL_san','W2M_san',
                'T2M_dav','QV2M_dav','TQL_dav','W2M_dav']
past_cov = TimeSeries.from_dataframe(df, value_cols=weather_cols)
future_cov = datetime_attribute_timeseries(df.index, attribute='day_of_week', one_hot=True) \
    .stack(datetime_attribute_timeseries(df.index, attribute='hour', one_hot=True)) \
    .stack(TimeSeries.from_dataframe(df, value_cols=['holiday','school']))

# Restrict training to 2019
train = series['2019-01-01':'2019-12-31']
test = series['2020-01-01':]

# Slice covariates
past_cov_train = past_cov.slice_intersect(train)
past_cov_test = past_cov.slice_intersect(test)
future_cov_train = future_cov.slice_intersect(train)
future_cov_test = future_cov.slice_intersect(test)

# Scaling
scaler_target = Scaler()
scaler_cov = Scaler()

train_scaled = scaler_target.fit_transform(train)
test_scaled = scaler_target.transform(test)
past_cov_scaled = scaler_cov.fit_transform(past_cov)
future_cov_scaled = scaler_cov.transform(future_cov)

past_cov_train_scaled = past_cov_scaled.slice_intersect(train_scaled)
future_cov_train_scaled = future_cov_scaled.slice_intersect(train_scaled)

# N-BEATS model
model = NBEATSModel(
    input_chunk_length=168,  # one week
    output_chunk_length=24,  # one day
    n_epochs=20,
    random_state=3363
)

model.fit(
    train_scaled,
    past_covariates=past_cov_train_scaled,
    future_covariates=future_cov_train_scaled,
    verbose=True
)

# Sliding window forecasts
forecast_scaled = model.historical_forecasts(
    series=train_scaled,
    past_covariates=past_cov_scaled,
    future_covariates=future_cov_scaled,
    start='2019-10-01',
    forecast_horizon=24,
    stride=24,
    retrain=False,
    verbose=True
)

# Inverse transform forecast
forecast = scaler_target.inverse_transform(forecast_scaled)

# Evaluate
mae = mean_absolute_error(test, forecast)
mape = mean_absolute_percentage_error(test, forecast) * 100
print(f"MAE: {mae:.2f}, MAPE: {mape:.2f}%")

# Plot
plt.figure(figsize=(14,6))
train.plot(label='Train', lw=1, color='black')
test.plot(label='Test', lw=1, color='gray')
forecast.plot(label='N-BEATS Forecast', lw=2, color='blue')
plt.xlabel('Date')
plt.ylabel('Load (MW)')
plt.title('Short-Term Electric Load Forecasting (N-BEATS) with Covariates')
plt.legend()
plt.savefig('electric_load_forecast_nbeats_refined.png')
plt.show()


ERROR:darts.timeseries:KeyError: 'Not all components found in time index.'


KeyError: 'Not all components found in time index.'

In [9]:
import pandas as pd
import matplotlib.pyplot as plt
from darts import TimeSeries
from darts.models import NBEATSModel
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error
from darts.dataprocessing.transformers import Scaler
from darts.utils.timeseries_generation import datetime_attribute_timeseries

# Load and preprocess data
df = pd.read_csv('continuous dataset.csv')
df['datetime'] = pd.to_datetime(df['datetime'])
df = df.set_index('datetime').asfreq('h').sort_index()

# Ensure time index covers expected range
print(f"Data covers {df.index.min()} to {df.index.max()}")

# Target series
series = TimeSeries.from_dataframe(df, value_cols='nat_demand')

# Covariates: weather (past) and calendar (future)
weather_cols = ['T2M_toc','QV2M_toc','TQL_toc','W2M_toc',
                'T2M_san','QV2M_san','TQL_san','W2M_san',
                'T2M_dav','QV2M_dav','TQL_dav','W2M_dav']
past_cov = TimeSeries.from_dataframe(df, value_cols=weather_cols)
future_cov = datetime_attribute_timeseries(series.time_index, attribute='day_of_week', one_hot=True) \
    .stack(datetime_attribute_timeseries(series.time_index, attribute='hour', one_hot=True)) \
    .stack(TimeSeries.from_dataframe(df, value_cols=['holiday','school']))

# Use slice with slice_intersect for safe date selection
train = series.slice(pd.Timestamp('2019-01-01'), pd.Timestamp('2019-12-31'))
test = series.slice(pd.Timestamp('2020-01-01'), pd.Timestamp('2020-12-31'))

# Slice covariates
past_cov_train = past_cov.slice_intersect(train)
past_cov_test = past_cov.slice_intersect(test)
future_cov_train = future_cov.slice_intersect(train)
future_cov_test = future_cov.slice_intersect(test)

# Scaling
scaler_target = Scaler()
scaler_cov = Scaler()

train_scaled = scaler_target.fit_transform(train)
test_scaled = scaler_target.transform(test)
past_cov_scaled = scaler_cov.fit_transform(past_cov)
future_cov_scaled = scaler_cov.fit_transform(future_cov)

past_cov_train_scaled = past_cov_scaled.slice_intersect(train_scaled)
future_cov_train_scaled = future_cov_scaled.slice_intersect(train_scaled)

# N-BEATS model
model = NBEATSModel(
    input_chunk_length=168,
    output_chunk_length=24,
    n_epochs=20,
    random_state=3363
)

model.fit(
    train_scaled,
    past_covariates=past_cov_train_scaled,
    verbose=True
)

# Sliding window forecasts
forecast_scaled = model.historical_forecasts(
    series=train_scaled,
    past_covariates=past_cov_scaled,
    future_covariates=future_cov_scaled,
    start=pd.Timestamp('2019-10-01'),
    forecast_horizon=24,
    stride=24,
    retrain=False,
    verbose=True
)

# Inverse transform forecast
forecast = scaler_target.inverse_transform(forecast_scaled)

# Evaluate
mae = mean_absolute_error(test, forecast)
mape = mean_absolute_percentage_error(test, forecast) * 100
print(f"MAE: {mae:.2f}, MAPE: {mape:.2f}%")

# Plot
plt.figure(figsize=(14,6))
train.plot(label='Train', lw=1, color='black')
test.plot(label='Test', lw=1, color='gray')
forecast.plot(label='N-BEATS Forecast', lw=2, color='blue')
plt.xlabel('Date')
plt.ylabel('Load (MW)')
plt.title('Short-Term Electric Load Forecasting (N-BEATS) with Covariates')
plt.legend()
plt.savefig('electric_load_forecast_nbeats_refined.png')
plt.show()


Data covers 2015-01-03 01:00:00 to 2020-06-27 00:00:00


INFO:pytorch_lightning.utilities.rank_zero:GPU available: False, used: False
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:HPU available: False, using: 0 HPUs
INFO:pytorch_lightning.callbacks.model_summary:
  | Name            | Type             | Params | Mode 
-------------------------------------------------------------
0 | criterion       | MSELoss          | 0      | train
1 | train_criterion | MSELoss          | 0      | train
2 | val_criterion   | MSELoss          | 0      | train
3 | train_metrics   | MetricCollection | 0      | train
4 | val_metrics     | MetricCollection | 0      | train
5 | stacks          | ModuleList       | 23.2 M | train
-------------------------------------------------------------
23.2 M    Trainable params
14.4 K    Non-trainable params
23.2 M    Total params
92.914    Total estimated model params size (MB)
396       Modules in train mode
0         Modules in eval mode


Training: |          | 0/? [00:00<?, ?it/s]

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=20` reached.
ERROR:darts.models.forecasting.torch_forecasting_model:ValueError: This model has been trained without `historic_future_covariates`; No `historic_future_covariates` should be provided for prediction.


ValueError: This model has been trained without `historic_future_covariates`; No `historic_future_covariates` should be provided for prediction.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from darts import TimeSeries
from darts.models import RNNModel
from darts.metrics import rmse
from darts.dataprocessing.transformers import Scaler

# Load continuous dataset
df = pd.read_csv("continuous dataset.csv")
df["datetime"] = pd.to_datetime(df["datetime"])
df_panama = df[["datetime", "nat_demand"]].rename(columns={"datetime": "timestamp", "nat_demand": "Load_MW"})
df_panama = df_panama.sort_values("timestamp").reset_index(drop=True)

# Convert to Darts TimeSeries
series = TimeSeries.from_dataframe(df_panama, "timestamp", "Load_MW")

# Scale series
scaler = Scaler()
series_scaled = scaler.fit_transform(series)

# Train/validation split (last week as validation)
train, val = series_scaled[:-24*7], series_scaled[-24*7:]

# LSTM model
model = RNNModel(
    model="LSTM",
    input_chunk_length=168,
    output_chunk_length=24,
    training_length=168,
    n_epochs=200,
    hidden_dim=32,
    dropout=0.1,
    random_state=42
)

model.fit(train, verbose=True)

# Forecast
pred_scaled = model.predict(len(val))
pred = scaler.inverse_transform(pred_scaled)
val_unscaled = scaler.inverse_transform(val)

# Plot forecast
plt.figure(figsize=(12, 4))
series[-24*7:].plot(label="Observed", lw=1, color="gray")
pred.plot(label="LSTM Forecast", lw=2, color="black")
plt.legend()
plt.title(f"Panama LSTM Load Forecast (RMSE: {rmse(val_unscaled, pred):.2f})")
plt.tight_layout()
plt.show()


/usr/local/lib/python3.11/dist-packages/torch/nn/modules/rnn.py:123: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.1 and num_layers=1
  warnings.warn(
INFO:pytorch_lightning.utilities.rank_zero:GPU available: False, used: False
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:HPU available: False, using: 0 HPUs
INFO:pytorch_lightning.callbacks.model_summary:
  | Name            | Type             | Params | Mode 
-------------------------------------------------------------
0 | criterion       | MSELoss          | 0      | train
1 | train_criterion | MSELoss          | 0      | train
2 | val_criterion   | MSELoss          | 0      | train
3 | train_metrics   | MetricCollection | 0      | train
4 | val_metrics     | MetricCollection | 0      | train
5 | rnn             | LSTM             | 4.5 K  | train
6 |

Training: |          | 0/? [00:00<?, ?it/s]